# 07 - BERTopic sobre el subcorpus de salud

In [8]:
# ============================================================
# CELL 0 - CONFIG
# ============================================================
from pathlib import Path

DATA_PROCESSED = Path(r'C:\Users\afpue\Documents\GitHub\icare\kMetodo\resultadosComportamiento')

NR_TOPICS_GRID  = [5, 10, 15, 20, "auto"]
NR_TOPICS_FINAL = 10
MIN_TOPIC_SIZE  = 5

print('[CONFIG] OK')
print(f'  DATA_PROCESSED : {DATA_PROCESSED.resolve()}')


[CONFIG] OK
  DATA_PROCESSED : C:\Users\afpue\Documents\GitHub\icare\kMetodo\resultadosComportamiento


In [9]:
# ============================================================
# CELL 0b - COMPATIBILIDAD NUMPY 2.x / TENSORFLOW
# ============================================================
import sys, types, importlib.machinery

if 'tensorflow' not in sys.modules:
    def _fake_mod(name):
        m = types.ModuleType(name)
        m.__spec__    = importlib.machinery.ModuleSpec(name, loader=None)
        m.__path__    = []
        m.__package__ = name
        m.__version__ = '0.0.0'
        return m
    for _n in [
        'tensorflow', 'tensorflow.python', 'tensorflow.python.eager',
        'tensorflow.python.framework', 'tensorflow.python.client',
        'tensorflow.python.util', 'tensorflow.python.ops',
        'tensorflow.core', 'tensorflow.keras', 'tensorflow.keras.layers',
        'tensorflow.keras.models', 'tensorflow.keras.callbacks',
        'tensorflow.keras.losses', 'tensorflow.keras.optimizers',
        'tensorflow.compat', 'tensorflow.compat.v1', 'tensorflow.compat.v2',
    ]:
        sys.modules[_n] = _fake_mod(_n)
    print('[COMPAT] tensorflow reemplazado con modulos ficticios')
else:
    print('[COMPAT] tensorflow ya estaba cargado, sin cambios')


[COMPAT] tensorflow ya estaba cargado, sin cambios


In [10]:
# ============================================================
# CELL 1 - IMPORTS Y CARGA
# ============================================================
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

corpus = pd.read_parquet(DATA_PROCESSED / 'corpus_cleaned.parquet')

tweets_etiquetados = pd.read_parquet(DATA_PROCESSED / 'comportamiento_etiquetados_final.parquet')

tweets_comp  = tweets_etiquetados[tweets_etiquetados['etiqueta_comportamiento'] == 1].copy()

df_comp  = tweets_comp.merge(
    corpus[['id_doc', 'Texto_limpio']],
    on='id_doc', how='left'
)

print(f'Corpus completo    : {len(corpus):,} tweets')
print(f'Subcorpus comp.    : {len(df_comp):,} tweets')
df_comp.head(2)


Corpus completo    : 151,424 tweets
Subcorpus comp.    : 2,804 tweets


,chunk_id,id_doc,texto_chunk,Comp_Desinfectante,Comp_Distanciamiento,Comp_Etiqueta_tos_codo,Comp_Etiqueta_tos_panuelo,Comp_Evitar_reuniones,Comp_Lavado_manos,Comp_Mascarilla,score_max,subcat_max,etiqueta_comportamiento,categoria_detectada,Texto_limpio
0,17_1,17,A esas horas el covid19 se queda en la casa mi...,0.037909,0.313174,0.314862,0.321228,0.509031,0.192823,0.118177,0.509031,Comp_Evitar_reuniones,1,Comp_Evitar_reuniones,A esas horas el covid19 se queda en la casa mi...
1,67_1,67,Los vendedores de balones de oxígeno y las pla...,0.253668,0.146621,0.448458,0.501524,0.200239,0.224089,0.342172,0.501524,Comp_Etiqueta_tos_panuelo,1,Comp_Etiqueta_tos_panuelo,Los vendedores de balones de oxígeno y las pla...


In [11]:
# ============================================================
# CELL 2 - CARGAR Y FILTRAR EMBEDDINGS
# ============================================================
embeddings_all = np.load(DATA_PROCESSED / 'tweet_embeddings.npy')
print(f'Embeddings totales  : {embeddings_all.shape}')

mask_comp        = tweets_etiquetados['etiqueta_comportamiento'] == 1
embeddings_comp  = embeddings_all[mask_comp.values]

print(f'Embeddings de comp. : {embeddings_comp.shape}')
assert len(embeddings_comp) == len(df_comp), (
    f"Mismatch: {len(embeddings_comp)} embeddings vs {len(df_comp)} tweets"
)
print('OK - embeddings alineados correctamente')


Embeddings totales  : (151424, 768)
Embeddings de comp. : (2804, 768)
OK - embeddings alineados correctamente


In [12]:
# ============================================================
# CELL 3 - STOPWORDS Y VECTORIZADOR
# ============================================================
nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])
spanish_stopwords = list(nlp.Defaults.stop_words)

twitter_sw = [
    'rt', 'http', 'https', 'co', 'amp', 'via', 'q', 'xq', 'x',
    'si', 'ya', 'asi', 'tan', 'ser', 'hay', 'ver', 'hoy',
]
all_stopwords = list(set(spanish_stopwords + twitter_sw))

vectorizer_model = CountVectorizer(
    stop_words=all_stopwords,
    max_features=5000,
    min_df=2,
    ngram_range=(1, 2),
    strip_accents=None
)

documents = df_comp['Texto_limpio'].fillna('').tolist()
print(f'Documentos para BERTopic: {len(documents):,}')


Documentos para BERTopic: 2,804


In [13]:
# ============================================================
# CELL 4 - FUNCIONES DE COHERENCIA
# ============================================================

def get_topic_words(topic_model, top_n=10):
    topics_words = []
    for topic_id, word_scores in topic_model.get_topics().items():
        if topic_id == -1:
            continue
        words = [word for word, _ in word_scores[:top_n]]
        topics_words.append(words)
    return topics_words


def compute_bertopic_coherence(topic_model, documents, vectorizer_model, top_n_words=10):
    analyzer       = vectorizer_model.build_analyzer()
    tokenized_docs = [analyzer(doc) for doc in documents]
    dictionary     = Dictionary(tokenized_docs)
    dictionary.filter_extremes(no_below=5, no_above=0.9)
    corpus_bow     = [dictionary.doc2bow(doc) for doc in tokenized_docs]
    topics_words   = get_topic_words(topic_model, top_n=top_n_words)
    cm = CoherenceModel(
        topics=topics_words, texts=tokenized_docs,
        dictionary=dictionary, corpus=corpus_bow, coherence='c_v'
    )
    return cm.get_coherence(), cm.get_coherence_per_topic()


In [14]:
# ============================================================
# CELL 5 - BUSQUEDA EN GRILLA DE nr_topics
# ============================================================
results = []

for nr in NR_TOPICS_GRID:
    print(f'\n--- nr_topics={nr} ---')
    tm = BERTopic(
        vectorizer_model=vectorizer_model,
        nr_topics=nr,
        min_topic_size=MIN_TOPIC_SIZE,
        calculate_probabilities=False,
        verbose=False,
        language="spanish"
    )
    topics, _ = tm.fit_transform(documents, embeddings=embeddings_comp)
    new_topics = tm.reduce_outliers(
        documents, topics, strategy="c-tf-idf", embeddings=embeddings_comp
    )
    tm.update_topics(documents, topics=new_topics, vectorizer_model=vectorizer_model)

    coherence, _ = compute_bertopic_coherence(tm, documents, vectorizer_model)
    n_outliers   = sum(1 for t in tm.topics_ if t == -1)
    results.append({
        "nr_topics"     : nr,
        "n_topics_final": len([t for t in tm.get_topics() if t != -1]),
        "coherence_c_v" : coherence,
        "pct_outliers"  : n_outliers / len(documents)
    })
    r = results[-1]
    print(f'  Topicos: {r["n_topics_final"]}  Coh: {r["coherence_c_v"]:.4f}  Outliers: {r["pct_outliers"]:.2%}')

df_grid = pd.DataFrame(results)
print()
print(df_grid.sort_values("coherence_c_v", ascending=False).to_string(index=False))



--- nr_topics=5 ---


2026-06-08 22:45:34,995 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


  Topicos: 4  Coh: 0.3719  Outliers: 0.00%

--- nr_topics=10 ---


2026-06-08 22:45:45,805 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


  Topicos: 9  Coh: 0.4272  Outliers: 0.00%

--- nr_topics=15 ---


...

  Topicos: 14  Coh: 0.4524  Outliers: 0.00%

--- nr_topics=20 ---


2026-06-08 22:46:07,131 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


  Topicos: 19  Coh: 0.4796  Outliers: 0.00%

--- nr_topics=auto ---


2026-06-08 22:46:26,499 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


  Topicos: 3  Coh: 0.3732  Outliers: 0.00%

nr_topics  n_topics_final  coherence_c_v  pct_outliers
       20              19       0.479577           0.0
       15              14       0.452397           0.0
       10               9       0.427243           0.0
     auto               3       0.373176           0.0
        5               4       0.371917           0.0


In [15]:
# ============================================================
# CELL 6 - MODELO FINAL
# ============================================================
topic_model = BERTopic(
    vectorizer_model=vectorizer_model,
    nr_topics=NR_TOPICS_FINAL,
    min_topic_size=MIN_TOPIC_SIZE,
    calculate_probabilities=False,
    verbose=True,
    language="spanish"
)

topics, probs = topic_model.fit_transform(documents, embeddings=embeddings_comp)
new_topics    = topic_model.reduce_outliers(
    documents, topics, strategy="c-tf-idf", embeddings=embeddings_comp
)
topic_model.update_topics(documents, topics=new_topics, vectorizer_model=vectorizer_model)

tm_res = topic_model.get_topic_info()
print(tm_res.head(10))


2026-06-08 22:46:29,373 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-08 22:46:38,840 - BERTopic - Dimensionality - Completed ✓
2026-06-08 22:46:38,840 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-08 22:46:38,938 - BERTopic - Cluster - Completed ✓
2026-06-08 22:46:38,938 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-06-08 22:46:39,068 - BERTopic - Representation - Completed ✓
2026-06-08 22:46:39,068 - BERTopic - Topic reduction - Reducing number of topics
2026-06-08 22:46:39,084 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-08 22:46:39,206 - BERTopic - Representation - Completed ✓
2026-06-08 22:46:39,206 - BERTopic - Topic reduction - Reduced number of topics from 89 to 10
2026-06-08 22:46:39,295 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure tha

   Topic  Count                                               Name  \
0      0   1114                    0_covid19_casa_medidas_personas   
1      1    926                        1_covid19_covid_covid 19_19   
2      2    341                        2_manos_agua_covid19_lavado   
3      3    184          3_mascarillas_mascarilla_covid19_máscaras   
4      4    127  4_social_distanciamiento_distanciamiento socia...   
5      5     41                            5_pa_pan_covid19_cocina   
6      6     33      6_basura_personas_guantes_aislamiento covid19   
7      7     20             7_cara_tengas_vamosaganarnoslavida_vas   
8      8     18                  8_humo_inhalar_pedo_respiratorias   

                                      Representation  \
0  [covid19, casa, medidas, personas, evitar, sal...   
1  [covid19, covid, covid 19, 19, gripe, virus, c...   
2  [manos, agua, covid19, lavado, jabón, lavado m...   
3  [mascarillas, mascarilla, covid19, máscaras, p...   
4  [social, distanc

In [16]:
# ============================================================
# CELL 7 - TABLA DE RESULTADOS FINAL
# ============================================================
coh_global, coh_per_topic = compute_bertopic_coherence(
    topic_model=topic_model,
    documents=documents,
    vectorizer_model=vectorizer_model,
    top_n_words=10
)
print(f'Coherencia global (c_v): {coh_global:.4f}')

topic_ids    = [t for t in topic_model.get_topics() if t != -1]
coherence_df = pd.DataFrame({"Topic": topic_ids, "Coherence_c_v": coh_per_topic})

total_docs  = tm_res["Count"].sum()
top         = tm_res.sort_values("Count", ascending=False).head(20).copy()
top["Keywords"]   = top["Representation"].apply(lambda ws: ", ".join(ws))
top["Porcentaje"] = (top["Count"] / total_docs * 100).round(2)
top = top.merge(coherence_df, on="Topic", how="left")

final_table = top[["Topic", "Count", "Porcentaje", "Coherence_c_v", "Keywords"]]
pd.set_option("display.max_colwidth", None)
print(final_table.to_string(index=False))


Coherencia global (c_v): 0.4543
 Topic  Count  Porcentaje  Coherence_c_v                                                                                                                                  Keywords
     0   1114       39.73       0.424055                                            covid19, casa, medidas, personas, evitar, salir, aislamiento, confinamiento, cuarentena, casos
     1    926       33.02       0.331630                                                   covid19, covid, covid 19, 19, gripe, virus, contagio, respiratoria, pacientes, síntomas
     2    341       12.16       0.637763                                           manos, agua, covid19, lavado, jabón, lavado manos, prevenir, lavarse, agua jabón, lavarse manos
     3    184        6.56       0.439152                                     mascarillas, mascarilla, covid19, máscaras, protección, salud, madrid, guantes, obligatorio, personal
     4    127        4.53       0.469845 social, distanciamiento, distanc

In [17]:
# ============================================================
# CELL 8 - ASIGNAR TOPICO Y GUARDAR
# ============================================================
import pickle

df_comp = df_comp.copy()
df_comp['topico'] = topic_model.topics_

coherence_table = (
    tm_res[tm_res["Topic"] != -1].merge(coherence_df, on="Topic", how="left")
)

ruta_excel = DATA_PROCESSED / 'bertopic_comportamiento_resultados.xlsx'
with pd.ExcelWriter(ruta_excel, engine='openpyxl') as writer:
    final_table.to_excel(writer,     sheet_name='Top_topicos',          index=False)
    coherence_table.to_excel(writer, sheet_name='Coherencia_por_topico',index=False)
    df_grid.to_excel(writer,         sheet_name='Grid_nr_topics',       index=False)
    df_comp[['id_doc', 'topico', 'categoria_detectada']].to_excel(
        writer, sheet_name='Tweets_por_topico', index=False
    )
print('[GUARDADO] bertopic_comportamiento_resultados.xlsx')

df_comp.to_parquet(DATA_PROCESSED / 'comportamiento_tweets_bertopic.parquet', index=False)
print('[GUARDADO] comportamiento_tweets_bertopic.parquet')

print()
print('Notebook 07 completado.')
print('Siguiente -> 08_extraer_frecuencias_POS.ipynb')


[GUARDADO] bertopic_comportamiento_resultados.xlsx
[GUARDADO] comportamiento_tweets_bertopic.parquet

Notebook 07 completado.
Siguiente -> 08_extraer_frecuencias_POS.ipynb
